# E3 · Group-growth ablations

## 1. Purpose and design

E3 varies the response vector and covariance used to construct candidate groups. VGMC uses the current response and complete covariance. Frozen uses the initial response. Diag uses diagonal covariance during growth. Frozen+Diag combines those settings. The four arms share the initial library, complete-covariance admission scores and joint coefficient and allocation optimization.

The upper panels show reference-state variance reduction relative to the common initial design. The lower panels show the fraction of added groups with allocation above `1e-12`. Curves use recorded addition counts. The Random reference uses five seeds. Its band shows the median and minimum–maximum range at each recorded count.

The molecular Hamiltonians use the authors' fermionic files and the Bravyi–Kitaev transform. [Hamiltonian sources](../docs/sources.md#molecular-hamiltonians)

## 2. Imports and settings

`MODE = "analysis_only"` processes the supplied records. `MODE = "full"` executes the configured tasks in `RUN` and analyzes their commits. `MODE = "render"` loads an existing analysis. `RUN_MODE` selects fresh execution, compatible resume, verified reuse or automatic selection.


In [ ]:
from pathlib import Path
import logging

from IPython.display import Image, display
from vgmc import plots
from vgmc.config import E3Config
from vgmc.data import read_json
from vgmc.workflow import analyze, load_analysis, render, run_experiment

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "DATA_MANIFEST.json").is_file())
MODE = "analysis_only"
RUN_MODE = "fresh"
MOLECULES = ['LiH', 'BeH2', 'H2O', 'NH3']
RUN = ROOT / "outputs" / "E3" / "run-001"
OUTPUT = ROOT / "outputs" / "E3" / "report"
ANALYSIS = OUTPUT / "analysis"
CONFIG = E3Config.from_mapping(
    read_json(ROOT / "configs" / "E3.json"))
logging.basicConfig(level=logging.INFO, format="%(message)s")
ARMS = ["A", "B", "C", "D"]
WORKERS = CONFIG.workers
RANDOM_REFERENCE = True


## 3. Run, collect and display

The analysis stores the selected rows and source hashes. Rendering reads that saved analysis.


In [ ]:
if MODE == "full":
    run_experiment("E3", ROOT, RUN, molecules=MOLECULES,
                   arms=ARMS, workers=WORKERS, random_reference=RANDOM_REFERENCE,
                   config=CONFIG, mode=RUN_MODE, allow_compute=True)
    analyze("E3", ROOT, ANALYSIS, run=RUN, molecules=MOLECULES)
elif MODE == "analysis_only":
    analyze("E3", ROOT, ANALYSIS, molecules=MOLECULES)
elif MODE != "render":
    raise ValueError("MODE must be analysis_only, full or render")

results = load_analysis("E3", ANALYSIS)
artifacts = render("E3", ANALYSIS, OUTPUT)


In [ ]:
endpoints = plots.summarize_e3(results)
display(endpoints[["molecule", "method", "seed", "added", "gain_pct", "active_pct"]])
for path in artifacts:
    if path.suffix == ".png":
        display(Image(filename=str(path)))
